In [1]:
import numpy as np
import pandas as pd

In [2]:
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [3]:
X_train = train_data.iloc[:, 1:].values
y_train = train_data.iloc[:, 0].values

In [4]:
X_test = test_data.values

In [5]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [6]:
y_onehot = np.zeros((len(y_train), 10))

for i in range(len(y_train)):
    y_onehot[i, y_train[i]] = 1

In [7]:
class DigitsClassify:

    def __init__(self, inp, hid, out):
        self.w1 = np.random.randn(inp, hid) * 0.01
        self.b1 = np.zeros((1, hid))

        self.w2 = np.random.randn(hid, out) * 0.01
        self.b2 = np.zeros((1, out))

    def sigmoid(self, x):
        return 1/(1+np.exp(-x))

    def sigmoid_grad(self, x):
        return x*(1-x)

    def softmax(self, x):
        x = x - np.max(x, axis=1, keepdims=True)
        e = np.exp(x)
        return e/np.sum(e, axis=1, keepdims=True)

    def forward(self, x):
        self.h = self.sigmoid(np.dot(x,self.w1)+self.b1)
        self.o = self.softmax(np.dot(self.h,self.w2)+self.b2)
        return self.o

    def backward(self, x, y, lr):

        n = len(x)

        error = self.o - y
        dw2 = np.dot(self.h.T,error)/n
        db2 = np.sum(error,axis=0,keepdims=True)/n

        err_h = np.dot(error,self.w2.T)*self.sigmoid_grad(self.h)
        dw1 = np.dot(x.T,err_h)/n
        db1 = np.sum(err_h,axis=0,keepdims=True)/n

        self.w2 -= lr*dw2
        self.b2 -= lr*db2
        self.w1 -= lr*dw1
        self.b1 -= lr*db1

    def train(self,x,y,epochs,lr):

        for i in range(epochs):

            self.forward(x)
            self.backward(x,y,lr)

            if (i+1)%50==0:
                loss = -np.mean(np.sum(y*np.log(self.o+1e-10),axis=1))
                print("epoch",i+1,"loss =",round(loss,4))

    def predict(self,x):
        out = self.forward(x)
        return np.argmax(out,axis=1)

In [8]:
model = DigitsClassify(X_train.shape[1],50,10)
model.train(X_train,y_onehot,epochs=200,lr=0.1)


epoch 50 loss = 2.2971
epoch 100 loss = 2.2873
epoch 150 loss = 2.257
epoch 200 loss = 2.1724


In [9]:
train_pred = model.predict(X_train)
acc = np.mean(train_pred==y_train)
print("Training accuracy:",acc)

test_pred = model.predict(X_test)
print("Test predictions:")
print(test_pred[:20])

Training accuracy: 0.3878809523809524
Test predictions:
[0 0 1 1 1 7 0 3 0 3 3 1 7 0 7 0 3 1 1 0]
